# Contrast sensitivity: analysis

Campbell & Robson (1968). Reads one or more run folders (`ANALYSIS_RUN`, several joined with the OS path separator; otherwise every run under `../data`). Thresholds come from the realised contrast of the drawn images. Definitions and the fitted function: `docs/methods.md`, Analysis. A run from `--mode simulate` is a toy observer, not data.

In [ ]:
import glob
import os
from pathlib import Path

import matplotlib
import pandas as pd

matplotlib.use('Agg')
import matplotlib.pyplot as plt

from campbell_robson1968 import analysis

env = os.environ.get('ANALYSIS_RUN')
runs = env.split(os.pathsep) if env else sorted({str(Path(p).parent) for p in glob.glob('../data/**/*_trials.csv', recursive=True)})
trials = analysis.load_runs(runs)
simulated = any('rehearsal' in r for r in runs)
print(len(runs), 'run(s);', len(trials), 'rows;', 'SIMULATED (toy observer)' if simulated else 'sessions')
trials.groupby('outcome').size()

## Two-interval forced choice: thresholds and sensitivity per cell

In [ ]:
detect = trials[trials['outcome'].isin(['CORRECT', 'WRONG'])]
csf = analysis.csf_table(trials, n_boot=100) if len(detect) else pd.DataFrame()
csf

In [ ]:
if len(csf):
    fig, ax = plt.subplots(figsize=(5, 4))
    for w, g in csf.groupby('waveform'):
        err = [g['sensitivity'] - g['sensitivity_ci_low'], g['sensitivity_ci_high'] - g['sensitivity']]
        ax.errorbar(g['sf_cpd'], g['sensitivity'], yerr=err, fmt='o', capsize=3, label=w)
    ax.set(xscale='log', yscale='log', xlabel='Spatial frequency (c/deg)', ylabel='Contrast sensitivity (1 / threshold)',
           title='Simulated toy observer' if simulated else 'Contrast sensitivity')
    ax.legend(frameon=False)
    plt.show()

## Square / sine sensitivity ratio (paper: 4/pi = 1.273 above about 0.8 c/deg)

In [ ]:
analysis.square_sine_ratio(csf) if len(csf) else 'no 2IFC trials'

## Asked-for versus drawn contrast, and timing

In [ ]:
if len(detect):
    d = detect.assign(ratio=detect['contrast_realised'] / detect['contrast_requested'])
    print(d.groupby('cell')['ratio'].describe()[['mean', 'std', 'min', 'max']])
    print('trials whose field had a single code value:', int((detect['n_codes'] <= 1).sum()))
analysis.realised_timing(trials)

## Adjustment: settings per frequency

In [ ]:
adj = analysis.adjustment_table(trials) if (trials['outcome'] == 'SET').any() else pd.DataFrame()
if len(adj):
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(adj['sf_cpd'], adj['sensitivity'], 'o')
    ax.set(xscale='log', yscale='log', xlabel='Spatial frequency (c/deg)', ylabel='Contrast sensitivity (1 / setting)')
    plt.show()
adj